# Qwen LoRA Fine-Tuning for Q&A on CPU

This notebook fine-tunes a Qwen instruction model for Q&A using LoRA adapters. CPU training does not use QLoRA because 4-bit `bitsandbytes` quantization is normally CUDA-backed.

Default CPU setup:

- Base model: `Qwen/Qwen2.5-0.5B-Instruct`
- Dataset: `squad` converted into instruction-answer chat examples
- Method: LoRA using `peft`
- Training loop: manual PyTorch loop with `tqdm` and MLflow metrics
- Batch optimization: length-based batch sorting with dynamic padding

CPU fine-tuning will be slow. Keep `MAX_SAMPLES`, `MAX_SEQ_LENGTH`, and LoRA rank small for a first run.

## 1. Install Dependencies

Run this cell once in a fresh environment. Restart the kernel after installing if your notebook runtime asks for it.

In [1]:
# %pip install -U "transformers>=4.51.0" "accelerate>=0.34.0" "datasets>=2.20.0" "peft>=0.12.0" "python-dotenv>=1.0.0" "huggingface_hub>=0.24.0" mlflow torch

## 2. Imports and Configuration

`MAX_SAMPLES` is intentionally small because CPU fine-tuning is slow. Increase it only after confirming that the full notebook runs on your machine.

In [ ]:
import os

import mlflow
import torch
from datasets import load_dataset
from dotenv import load_dotenv
from peft import LoraConfig, get_peft_model
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import transformers
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
)

load_dotenv(".env", override=True)


def _transformers_version_tuple(version_string: str) -> tuple[int, int, int]:
    """Parse '4.51.2' / '4.51.0.dev0' into (major, minor, micro) for min-version checks."""
    parts = version_string.split("+")[0].split("rc")[0].split("a")[0].split("b")[0].split("dev")[0]
    nums = []
    for p in parts.replace("-", ".").split("."):
        if p.isdigit():
            nums.append(int(p))
        else:
            break
    while len(nums) < 3:
        nums.append(0)
    return nums[0], nums[1], nums[2]


def _assert_transformers_for_qwen3_moe(model_id: str) -> None:
    if "qwen3" not in model_id.lower():
        return
    if _transformers_version_tuple(transformers.__version__) < (4, 51, 0):
        raise RuntimeError(
            f"Qwen3 MoE ({model_id}) needs transformers>=4.51.0; you have {transformers.__version__}. "
            "Uncomment and run the install cell, restart the kernel, then retry."
        )


# Default matches the CPU-oriented intro above. Override: set BASE_MODEL in .env or the environment.
_default_base = "Qwen/Qwen2.5-0.5B-Instruct"
_base_model = (os.environ.get("BASE_MODEL") or _default_base).strip() or _default_base
_assert_transformers_for_qwen3_moe(_base_model)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

params = {
    "BASE_MODEL": _base_model,
    "DATASET_NAME": "squad",
    "OUTPUT_DIR": "./qwen-qa-lora-cpu",
    "ADAPTER_DIR": "./qwen-qa-lora-cpu/adapter",
    "TASK_NAME": "qwen_qa_lora_cpu_30b__",
    "MAX_SAMPLES": 100,
    "MAX_EVAL_SAMPLES": 32,
    "MAX_SEQ_LENGTH": 512,
    "BATCH_SIZE": 1,
    "GRADIENT_ACCUMULATION_STEPS": 4,
    "GROUP_BY_LENGTH": True,
    "LEARNING_RATE": 2e-4,
    "NUM_EPOCHS": 1,
    "CHECKPOINT_DIR": "./qwen-qa-lora-cpu/checkpoints",
    "CHECKPOINT_EVERY_STEPS": 100,
    "CHECKPOINT_MAX_KEEP": 3,
    "DEVICE": DEVICE,
    "DTYPE": DTYPE,
}

BASE_MODEL = params["BASE_MODEL"]
DATASET_NAME = params["DATASET_NAME"]
OUTPUT_DIR = params["OUTPUT_DIR"]
ADAPTER_DIR = params["ADAPTER_DIR"]
MAX_SAMPLES = params["MAX_SAMPLES"]
MAX_EVAL_SAMPLES = params["MAX_EVAL_SAMPLES"]
MAX_SEQ_LENGTH = params["MAX_SEQ_LENGTH"]
BATCH_SIZE = params["BATCH_SIZE"]
GRADIENT_ACCUMULATION_STEPS = params["GRADIENT_ACCUMULATION_STEPS"]
GROUP_BY_LENGTH = params["GROUP_BY_LENGTH"]
LEARNING_RATE = params["LEARNING_RATE"]
NUM_EPOCHS = params["NUM_EPOCHS"]
CHECKPOINT_DIR = params["CHECKPOINT_DIR"]
CHECKPOINT_EVERY_STEPS = params["CHECKPOINT_EVERY_STEPS"]
CHECKPOINT_MAX_KEEP = params["CHECKPOINT_MAX_KEEP"]

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"BASE_MODEL: {BASE_MODEL}")
print(f"transformers: {transformers.__version__}")
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU. This is for small experiments and will be slow.")
print(f"Model dtype: {DTYPE}")

BASE_MODEL: Qwen/Qwen3-30B-A3B-Instruct-2507
transformers: 5.8.1
Device: cpu
Running on CPU. This is for small experiments and will be slow.
Model dtype: torch.float32


## 3. Load and Format Q&A Data

SQuAD examples are converted to chat-style instruction data:

- User: context + question
- Assistant: answer

For your own data, create the same `text` column with a complete chat-formatted example.

In [ ]:
import sys
from pathlib import Path

_api = next(
    (p for p in (Path.cwd(), Path.cwd() / "api", Path.cwd() / "llm" / "api") if (p / "notebook_bootstrap.py").exists()),
    None,
)
if _api:
    sys.path.insert(0, str(_api))
    from notebook_bootstrap import load_environment

    load_environment()

mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000"))
mlflow.set_experiment(params["TASK_NAME"])

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"MLflow experiment: {params['TASK_NAME']}")

MLflow tracking URI: http://127.0.0.1:5000
MLflow experiment: qwen_qa_lora_cpu_30b_1_


In [4]:
try:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
except OSError as exc:
    raise OSError(
        "Could not download or load tokenizer/config from Hugging Face. "
        "Typical causes: no DNS/network to huggingface.co, VPN/proxy, or BASE_MODEL in .env "
        "pointing at a repo that cannot be fetched. Comment BASE_MODEL in .env to use the "
        f"notebook default. Current BASE_MODEL={BASE_MODEL!r}."
    ) from exc

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

train_raw = load_dataset(DATASET_NAME, split=f"train[:{MAX_SAMPLES}]")
eval_raw = load_dataset(DATASET_NAME, split=f"validation[:{MAX_EVAL_SAMPLES}]")
mlflow.log_artifact("data/train_dataset.parquet", artifact_path="datasets")
mlflow.log_artifact("data/test_dataset.parquet", artifact_path="datasets")

def format_qa_example(example):
    answer = example["answers"]["text"][0] if example["answers"]["text"] else "I do not know."
    user_prompt = (
        "Answer the question using only the context.\n\n"
        f"Context:\n{example['context']}\n\n"
        f"Question:\n{example['question']}"
    )
    messages = [
        {"role": "system", "content": "You are a helpful Q&A assistant."},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": answer},
    ]
    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    }

train_dataset = train_raw.map(format_qa_example, remove_columns=train_raw.column_names)
eval_dataset = eval_raw.map(format_qa_example, remove_columns=eval_raw.column_names)

print(train_dataset[0]["text"][:1_000])

config.json:   0%|          | 0.00/963 [00:00<?, ?B/s]

c:\Users\guslc\miniconda3\envs\llm\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\guslc\.cache\huggingface\hub\models--Qwen--Qwen3-30B-A3B-Instruct-2507. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

<|im_start|>system
You are a helpful Q&A assistant.<|im_end|>
<|im_start|>user
Answer the question using only the context.

Context:
Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.

Question:
To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?<|im_end|>
<|im_start|>assistant
Saint Bernadette Soubirous<|im_end|>



## 4. Tokenize Data and Sort Batches by Length

The examples are tokenized after chat formatting. The language modeling collator dynamically pads each batch, and `group_by_length=True` makes the trainer batch examples with similar token lengths together.

In [5]:
def tokenize_function(batch):
    tokenized = tokenizer(
        batch["text"],
        max_length=MAX_SEQ_LENGTH,
        truncation=True,
        padding=False,
    )
    tokenized["length"] = [len(input_ids) for input_ids in tokenized["input_ids"]]
    return tokenized

tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
)
tokenized_eval = eval_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

print("Train token length range:", min(tokenized_train["length"]), "-", max(tokenized_train["length"]))
print("Eval token length range:", min(tokenized_eval["length"]), "-", max(tokenized_eval["length"]))

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Train token length range: 160 - 434
Eval token length range: 188 - 232


## 5. Load Qwen and Add LoRA Adapters

On CPU this uses standard LoRA instead of QLoRA. The base model is loaded in full precision and frozen, while only the small LoRA adapter weights are trained.

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=DTYPE,
    trust_remote_code=True,
)
model.to(DEVICE)
model.config.use_cache = False

if DEVICE == "cuda":
    model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

## 6. Train and Log Metrics

This manual loop mirrors the style from `experiment.ipynb`: tqdm tracks completion, MLflow logs batch loss, and each epoch records validation loss plus token-level accuracy, precision, recall, and F1.

In [ ]:
def prepare_batch(batch):
    return {key: value.to(DEVICE) for key, value in batch.items() if key != "length"}

In [ ]:
def update_token_metric_counts(logits, labels, stats):
    # Causal LM predicts the next token, so compare logits[t] with labels[t + 1].
    predictions = logits[:, :-1, :].argmax(dim=-1)
    targets = labels[:, 1:]
    mask = targets != -100

    predictions = predictions[mask].detach().cpu()
    targets = targets[mask].detach().cpu()

    if targets.numel() == 0:
        return

    stats["correct"] += int((predictions == targets).sum().item())
    stats["total"] += int(targets.numel())

    for token_id in torch.unique(torch.cat([predictions, targets])).tolist():
        pred_mask = predictions == token_id
        target_mask = targets == token_id
        true_positive = int((pred_mask & target_mask).sum().item())
        predicted_count = int(pred_mask.sum().item())
        target_count = int(target_mask.sum().item())

        token_stats = stats["per_token"].setdefault(
            int(token_id),
            {"tp": 0, "predicted": 0, "target": 0},
        )
        token_stats["tp"] += true_positive
        token_stats["predicted"] += predicted_count
        token_stats["target"] += target_count

In [ ]:
def finalize_token_metrics(stats):
    if stats["total"] == 0:
        return {"accuracy": 0.0, "precision": 0.0, "recall": 0.0, "f1": 0.0}

    accuracy = stats["correct"] / stats["total"]
    weighted_precision = 0.0
    weighted_recall = 0.0
    weighted_f1 = 0.0

    for token_stats in stats["per_token"].values():
        support = token_stats["target"]
        if support == 0:
            continue

        precision = token_stats["tp"] / token_stats["predicted"] if token_stats["predicted"] else 0.0
        recall = token_stats["tp"] / support
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

        weight = support / stats["total"]
        weighted_precision += weight * precision
        weighted_recall += weight * recall
        weighted_f1 += weight * f1

    return {
        "accuracy": accuracy,
        "precision": weighted_precision,
        "recall": weighted_recall,
        "f1": weighted_f1,
    }

In [ ]:
import shutil


@torch.no_grad()
def evaluate(model, eval_loader):
    model.eval()
    total_loss = 0.0
    stats = {"correct": 0, "total": 0, "per_token": {}}

    for batch in eval_loader:
        batch = prepare_batch(batch)
        outputs = model(**batch)
        total_loss += outputs.loss.item()
        update_token_metric_counts(outputs.logits, batch["labels"], stats)

    metrics = finalize_token_metrics(stats)
    metrics["eval_loss"] = total_loss / max(len(eval_loader), 1)
    model.train()
    return metrics


train_data = tokenized_train.sort("length") if GROUP_BY_LENGTH else tokenized_train
eval_data = tokenized_eval.sort("length") if GROUP_BY_LENGTH else tokenized_eval

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=not GROUP_BY_LENGTH,
    collate_fn=data_collator,
)
eval_loader = DataLoader(
    eval_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)


def log_metric_now(key, value, step):
    try:
        mlflow.log_metric(key, value, step=step, synchronous=True)
    except TypeError:
        mlflow.log_metric(key, value, step=step)


def log_metrics_now(metrics, step):
    try:
        mlflow.log_metrics(metrics, step=step, synchronous=True)
    except TypeError:
        mlflow.log_metrics(metrics, step=step)


def save_training_checkpoint(epoch, global_step, optim_step, metrics=None, tag=""):
    parts = [f"e{epoch}", f"gs{global_step}", f"os{optim_step}"]
    if tag:
        parts.append(tag)
    dirname = "ckpt-" + "-".join(parts)
    path = os.path.join(CHECKPOINT_DIR, dirname)
    os.makedirs(path, exist_ok=True)
    model.save_pretrained(path)
    tokenizer.save_pretrained(path)
    state = {
        "epoch": epoch,
        "global_step": global_step,
        "optim_step": optim_step,
        "optimizer": optimizer.state_dict(),
    }
    if metrics is not None:
        state["metrics"] = {
            k: float(v) for k, v in metrics.items() if isinstance(v, (int, float))
        }
    torch.save(state, os.path.join(path, "training_state.pt"))
    print(f"Checkpoint saved: {path}")
    return path


def maybe_prune_checkpoints():
    if CHECKPOINT_MAX_KEEP <= 0:
        return
    if not os.path.isdir(CHECKPOINT_DIR):
        return
    dirs = sorted(
        (
            os.path.join(CHECKPOINT_DIR, d)
            for d in os.listdir(CHECKPOINT_DIR)
            if d.startswith("ckpt-") and os.path.isdir(os.path.join(CHECKPOINT_DIR, d))
        ),
        key=os.path.getmtime,
    )
    while len(dirs) > CHECKPOINT_MAX_KEEP:
        oldest = dirs.pop(0)
        shutil.rmtree(oldest, ignore_errors=True)
        print(f"Pruned old checkpoint: {oldest}")


created_run = mlflow.active_run() is None

if created_run:
    mlflow.start_run(run_name=params["TASK_NAME"])

print(f"MLflow run_id: {mlflow.active_run().info.run_id}")

try:
    mlflow.log_params({key: str(value) for key, value in params.items()})
    model.train()
    optimizer.zero_grad()
    global_step = 0
    optim_step = 0

    total_steps = NUM_EPOCHS * len(train_loader)
    with tqdm(total=total_steps, desc=f"Epoch [1/{NUM_EPOCHS}] - (Loss: N/A) - Steps") as pbar:
        for epoch in range(NUM_EPOCHS):
            running_loss = 0.0

            for step, batch in enumerate(train_loader):
                batch = prepare_batch(batch)
                outputs = model(**batch)
                loss = outputs.loss
                (loss / GRADIENT_ACCUMULATION_STEPS).backward()

                running_loss += loss.item()
                global_step += 1

                should_step = (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0
                is_last_step = step + 1 == len(train_loader)
                if should_step or is_last_step:
                    optimizer.step()
                    optimizer.zero_grad()
                    optim_step += 1
                    if (
                        CHECKPOINT_EVERY_STEPS > 0
                        and optim_step % CHECKPOINT_EVERY_STEPS == 0
                    ):
                        save_training_checkpoint(
                            epoch + 1,
                            global_step,
                            optim_step,
                            metrics=None,
                            tag="step",
                        )
                        maybe_prune_checkpoints()

                if global_step % 5 == 0:
                    avg_loss = running_loss / 5
                    pbar.set_description(
                        f"Epoch [{epoch + 1}/{NUM_EPOCHS}] - (Loss: {avg_loss:.3f}) - Steps"
                    )
                    log_metric_now("loss", avg_loss, step=global_step)
                    log_metric_now("train_loss", avg_loss, step=global_step)
                    print(f"Logged train_loss={avg_loss:.4f} at step {global_step}")
                    running_loss = 0.0

                pbar.update(1)

            metrics = evaluate(model, eval_loader)
            print(
                f"Epoch {epoch + 1} Metrics: "
                f"Eval Loss: {metrics['eval_loss']:.4f}, "
                f"Accuracy: {metrics['accuracy']:.4f}, "
                f"Precision: {metrics['precision']:.4f}, "
                f"Recall: {metrics['recall']:.4f}, "
                f"F1: {metrics['f1']:.4f}"
            )
            epoch_metrics = {
                "eval_loss": metrics["eval_loss"],
                "eval_accuracy": metrics["accuracy"],
                "eval_precision": metrics["precision"],
                "eval_recall": metrics["recall"],
                "eval_f1": metrics["f1"],
            }
            log_metrics_now(epoch_metrics, step=epoch + 1)
            print(f"Logged eval metrics to MLflow at epoch {epoch + 1}: {epoch_metrics}")
            save_training_checkpoint(
                epoch + 1,
                global_step,
                optim_step,
                metrics=metrics,
                tag=f"epoch{epoch + 1}",
            )
            maybe_prune_checkpoints()

    os.makedirs(ADAPTER_DIR, exist_ok=True)
    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    mlflow.log_artifacts(ADAPTER_DIR, artifact_path="model")
    mlflow.log_artifacts(OUTPUT_DIR, artifact_path="model")
    print("Finished Training")
finally:
    if created_run:
        mlflow.end_run()


## 7. Store the Adapter in MLflow and Hugging Face

Run this after the training cell finishes. It uploads the LoRA adapter in `ADAPTER_DIR` to Hugging Face and logs the Hugging Face repo URL plus the adapter files back to MLflow.

Add these values to `.env` in the same folder as this notebook:

```env
HF_TOKEN=your_huggingface_write_token
HF_REPO_ID=your-username/qwen-qa-lora-cpu
HF_PRIVATE=true
```

Use `HF_PRIVATE=false` if you want the model repository to be public.

In [ ]:
from huggingface_hub import HfApi, login

import sys
from pathlib import Path

_api = next(
    (p for p in (Path.cwd(), Path.cwd() / "api", Path.cwd() / "llm" / "api") if (p / "notebook_bootstrap.py").exists()),
    None,
)
if _api:
    sys.path.insert(0, str(_api))
    from notebook_bootstrap import load_environment

    load_environment()

hf_token = os.getenv("HF_TOKEN")
hf_repo_id = os.getenv("HF_REPO_ID")
hf_private = os.getenv("HF_PRIVATE", "true").lower() == "true"
hf_url = f"https://huggingface.co/{hf_repo_id}" if hf_repo_id else None

if not hf_token:
    raise RuntimeError("Set HF_TOKEN in .env with a Hugging Face token that has write permission.")

if not hf_repo_id:
    raise RuntimeError("Set HF_REPO_ID in .env, for example: your-username/qwen-qa-lora-cpu")

if not os.path.isdir(ADAPTER_DIR):
    raise FileNotFoundError(f"Adapter directory not found: {ADAPTER_DIR}. Train and save the adapter first.")

adapter_weight_files = ["adapter_model.safetensors", "adapter_model.bin"]
if not any(os.path.exists(os.path.join(ADAPTER_DIR, filename)) for filename in adapter_weight_files):
    raise FileNotFoundError(
        "No LoRA adapter weight file found. Expected adapter_model.safetensors or adapter_model.bin "
        f"inside {ADAPTER_DIR}. Run model.save_pretrained(ADAPTER_DIR) after training."
    )

model_card = f"""---
base_model: {BASE_MODEL}
library_name: peft
tags:
- qwen
- lora
- question-answering
- causal-lm
---

# Qwen Q&A LoRA Adapter

This repository contains a LoRA adapter fine-tuned for question answering.

## Base Model

`{BASE_MODEL}`

## Training Data

`{DATASET_NAME}` formatted as context-question-answer chat examples.

## MLflow

The adapter files and Hugging Face repo URL are also logged in MLflow under experiment `{params['TASK_NAME']}`.

## Usage

Load this adapter with `peft.PeftModel.from_pretrained` on top of the base model.
"""

with open(os.path.join(ADAPTER_DIR, "README.md"), "w", encoding="utf-8") as f:
    f.write(model_card)

login(token=hf_token, add_to_git_credential=False)

api = HfApi(token=hf_token)
api.create_repo(repo_id=hf_repo_id, repo_type="model", private=hf_private, exist_ok=True)
api.upload_folder(
    folder_path=ADAPTER_DIR,
    repo_id=hf_repo_id,
    repo_type="model",
    commit_message="Upload Qwen Q&A LoRA adapter",
)

if mlflow.active_run() is not None:
    mlflow.end_run()

with mlflow.start_run(run_name=f"{params['TASK_NAME']}_huggingface_publish"):
    mlflow.log_param("hf_repo_id", hf_repo_id)
    mlflow.log_param("hf_private", hf_private)
    mlflow.set_tag("huggingface_url", hf_url)
    mlflow.set_tag("base_model", BASE_MODEL)
    mlflow.log_artifacts(ADAPTER_DIR, artifact_path="huggingface_adapter")

print(f"Published adapter to: {hf_url}")
print("Logged Hugging Face repo metadata and adapter snapshot to MLflow.")

## 7. Save the LoRA Adapter

This saves only the LoRA adapter weights, not a full copy of the base model.

In [ ]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"Saved LoRA adapter to: {ADAPTER_DIR}")

## 8. Test the Fine-Tuned Adapter

Because the adapter is already loaded in this session, you can generate directly from `model`.

In [ ]:
def answer_question(context, question, max_new_tokens=128):
    model.eval()
    messages = [
        {"role": "system", "content": "You are a helpful Q&A assistant."},
        {
            "role": "user",
            "content": (
                "Answer the question using only the context.\n\n"
                f"Context:\n{context}\n\n"
                f"Question:\n{question}"
            ),
        },
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

context = "Qwen is a family of large language models developed by Alibaba Cloud. LoRA fine-tunes a model by training small adapter weights."
question = "What does LoRA train?"

print(answer_question(context, question))

In [ ]:
context = ""
question = "I have table a and table b and they have a common key id, how to join them?"

print(answer_question(context, question))

## 9. Load the Saved Adapter Later

Use this in a new session when doing inference from the saved adapter.

In [ ]:
# from peft import PeftModel

# tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)
# base_model = AutoModelForCausalLM.from_pretrained(
#     BASE_MODEL,
#     torch_dtype=DTYPE,
#     trust_remote_code=True,
# )
# base_model.to(DEVICE)
# model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
# model.eval()

In [ ]:
mlflow.end_run()